# GRC Security Intelligence - Semantic layer, Genie space and app grants

Runs after the medallion pipeline in the **One-time Setup** job:

1. Creates the Unity Catalog metric views `grc_gold.mv_grc_metrics` and `grc_gold.mv_vuln_metrics`.
2. Creates (or updates) the Genie space **GRC Security Assistant** from `src/genie/grc_security_assistant_space.json`.
3. Grants the GRC Control Center app service principal read access to the GRC schemas, CAN_VIEW on the pipeline and CAN_RUN on the Genie space.

Idempotent - safe to re-run.

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("warehouse_id", "")
dbutils.widgets.text("app_name", "")
dbutils.widgets.text("pipeline_id", "")
dbutils.widgets.text("as_of_date", "2026-10-27")

CAT = dbutils.widgets.get("catalog")
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
APP_NAME = dbutils.widgets.get("app_name")
PIPELINE_ID = dbutils.widgets.get("pipeline_id")
AS_OF = dbutils.widgets.get("as_of_date")
GENIE_TITLE = "GRC Security Assistant"
Q = f"`{CAT}`"
print(CAT, WAREHOUSE_ID, APP_NAME, PIPELINE_ID, AS_OF)

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {Q}.grc_gold.mv_grc_metrics
COMMENT 'Governed GRC control-test metrics (pass rate, exceptions, effectiveness) over row-level control test evidence. Shared semantic layer for AI/BI dashboards, Genie and Power BI. Effective >= 95% pass, Partially Effective 80-95%, Ineffective < 80%.'
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: '{Q}.grc_gold.control_test_evidence'
joins:
  - name: ctl
    source: '{Q}.grc_ref.controls'
    "on": source.control_id = ctl.control_id
  - name: bu
    source: '{Q}.grc_ref.business_units'
    "on": source.bu_id = bu.bu_id
dimensions:
  - name: Test Run Date
    expr: test_run_date
    comment: Control test run date (pipeline as_of_date)
  - name: Test ID
    expr: test_id
  - name: Control ID
    expr: control_id
  - name: Control Name
    expr: ctl.control_name
  - name: Control Domain
    expr: ctl.control_domain
  - name: Key Control
    expr: ctl.key_control
  - name: Business Unit
    expr: "coalesce(bu.bu_name, 'Unassigned')"
  - name: Severity
    expr: severity
  - name: Entity Type
    expr: entity_type
  - name: Source Table
    expr: source_table
measures:
  - name: Population
    expr: COUNT(1)
    comment: Population members tested
  - name: Exceptions
    expr: COUNT_IF(is_exception)
    comment: Population members failing the test
  - name: Critical Exceptions
    expr: COUNT_IF(is_exception AND severity = 'Critical')
  - name: Overdue Exceptions
    expr: COUNT_IF(is_exception AND due_date < test_run_date)
    comment: Exceptions past their remediation due date
  - name: Pass Rate Pct
    expr: "ROUND(100.0 * COUNT_IF(NOT is_exception) / NULLIF(COUNT(1), 0), 2)"
    comment: Passing members / population x 100
  - name: Effectiveness Band
    expr: "CASE WHEN 100.0 * COUNT_IF(NOT is_exception) / NULLIF(COUNT(1), 0) >= 95 THEN 'Effective' WHEN 100.0 * COUNT_IF(NOT is_exception) / NULLIF(COUNT(1), 0) >= 80 THEN 'Partially Effective' ELSE 'Ineffective' END"
    comment: Band for the aggregated pass rate
  - name: Controls Tested
    expr: COUNT(DISTINCT control_id)
$$""")
display(spark.sql(f"SELECT `Control Domain`, MEASURE(`Pass Rate Pct`) AS pass_rate, MEASURE(`Exceptions`) AS exceptions FROM {Q}.grc_gold.mv_grc_metrics GROUP BY ALL ORDER BY 1"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {Q}.grc_gold.mv_vuln_metrics
COMMENT 'Governed vulnerability management metrics: open findings, past-SLA, KEV exposure, MTTR and SLA compliance by business unit, tier, severity and environment. Scope matches grc_gold.vuln_sla_compliance: open findings plus findings remediated in the 90 days before as_of_date.'
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: '{Q}.grc_silver.vulnerability_finding'
joins:
  - name: bu
    source: '{Q}.grc_ref.business_units'
    "on": source.bu_id = bu.bu_id
filter: "is_open OR remediated_date >= date_sub(as_of_date, 90)"
dimensions:
  - name: Business Unit
    expr: "coalesce(bu.bu_name, 'Unassigned')"
  - name: Asset Tier
    expr: tier
  - name: Severity
    expr: severity
  - name: Environment
    expr: environment
  - name: Internet Facing
    expr: internet_facing
  - name: Is KEV
    expr: is_kev
  - name: First Found Week
    expr: "DATE_TRUNC('WEEK', first_found)"
measures:
  - name: Findings
    expr: COUNT(1)
  - name: Open Findings
    expr: COUNT_IF(is_open)
  - name: Open Past SLA
    expr: COUNT_IF(is_open AND is_overdue)
  - name: Open KEV Findings
    expr: COUNT_IF(is_open AND is_kev)
  - name: SLA Compliance Pct
    expr: "ROUND(100.0 * COUNT_IF(NOT sla_breached) / NULLIF(COUNT(1), 0), 2)"
    comment: Findings not breaching SLA / all findings x 100
  - name: MTTR Days
    expr: "ROUND(AVG(CASE WHEN NOT is_open THEN days_open END), 1)"
    comment: Mean days to remediate (remediated findings)
  - name: Hosts With Open Findings
    expr: COUNT(DISTINCT CASE WHEN is_open THEN hostname END)
$$""")
display(spark.sql(f"SELECT `Severity`, MEASURE(`Open Findings`) AS open_findings, MEASURE(`Open Past SLA`) AS past_sla FROM {Q}.grc_gold.mv_vuln_metrics GROUP BY ALL ORDER BY 1"))

In [0]:
import json, os
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
me = w.current_user.me().user_name

# Template lives next to this notebook in the bundle: src/genie/grc_security_assistant_space.json
tpl_path = os.path.abspath(os.path.join(os.getcwd(), "..", "genie", "grc_security_assistant_space.json"))
raw = open(tpl_path).read().replace("__CATALOG__", CAT).replace("__AS_OF_DATE__", AS_OF)
serialized = json.dumps(json.loads(raw))  # validate JSON

description = ("Natural-language Q&A over ACME Corp GRC security intelligence: control effectiveness, framework posture "
               "(NIST CSF 2.0, ISO 27001, SOC 2, CIS v8), vulnerabilities, identity risk, physical security, policies, "
               "exceptions, risk register and audit evidence.")

def find_space(title):
    token = None
    while True:
        q = {"page_size": 100}
        if token:
            q["page_token"] = token
        r = w.api_client.do("GET", "/api/2.0/genie/spaces", query=q)
        for s in r.get("spaces", []):
            if s.get("title") == title:
                return s["space_id"]
        token = r.get("next_page_token")
        if not token:
            return None

space_id = find_space(GENIE_TITLE)
if space_id:
    w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{space_id}",
                    body={"serialized_space": serialized, "warehouse_id": WAREHOUSE_ID, "description": description})
    print("Updated Genie space", space_id)
else:
    r = w.api_client.do("POST", "/api/2.0/genie/spaces",
                        body={"title": GENIE_TITLE, "description": description, "warehouse_id": WAREHOUSE_ID,
                              "parent_path": f"/Workspace/Users/{me}", "serialized_space": serialized})
    space_id = r["space_id"]
    print("Created Genie space", space_id)

In [0]:
sp = None
if APP_NAME:
    try:
        sp = w.apps.get(APP_NAME).service_principal_client_id
    except Exception as e:
        print(f"App {APP_NAME} not found yet ({e}); deploy the bundle, then re-run this task.")

if sp:
    spark.sql(f"GRANT USE CATALOG ON CATALOG {Q} TO `{sp}`")
    for s in ["grc_ref", "grc_bronze", "grc_silver", "grc_gold"]:
        spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {Q}.{s} TO `{sp}`")
    if PIPELINE_ID:
        w.api_client.do("PATCH", f"/api/2.0/permissions/pipelines/{PIPELINE_ID}",
                        body={"access_control_list": [{"service_principal_name": sp, "permission_level": "CAN_VIEW"}]})
    w.api_client.do("PATCH", f"/api/2.0/permissions/genie/{space_id}",
                    body={"access_control_list": [{"service_principal_name": sp, "permission_level": "CAN_RUN"}]})
    print(f"Granted app SP {sp}: UC read on GRC schemas, CAN_VIEW pipeline, CAN_RUN Genie {space_id}")

In [0]:
display(spark.sql(f"""
SELECT kpi_domain, kpi_id, kpi_name, kpi_value, unit, target_value, kpi_status
FROM {Q}.grc_gold.grc_kpi_summary ORDER BY kpi_domain, kpi_id"""))